# Big Notebook Part 10: Merging Real Neuron Data (Allen Cell Types)
Real data again: the **2,333 neurons** from the Allen Cell Types Database. This time the information is spread over **four tables**, just like in a real research database. The setup cell builds them from `allen_cells.csv`:

| Table | One row per … | Key column |
|---|---|---|
| `zellen` | neuron (species, dendrite type, layer, brain area) | `cell_id`, `donor_id`, `area` |
| `ephys` | neuron **with** a measured firing rate (electrophysiology) | `specimen` (= `cell_id`!) |
| `spender` | **human** donor (age, sex, disease) | `id` (= `donor_id`!) |
| `areale` | brain area (full name, system) – a small, **incomplete** lookup table | `area` |

Your job: combine them with `concat`, `merge` and `join` – and find out what gets lost along the way.

10 tasks, about 60–75 minutes. Focus: Part 10, plus your core patterns (`groupby`, function + `apply`, loop).
- 🔍 = Spot the bug
- 🧠 = DataFrame, Series or a single value?
- Before each merge, ask yourself: **Which rows should survive?**

---
## Setup
Run this cell once – it builds the four tables.

In [1]:
import pandas as pd

# Path to the data file – adjust to where you saved allen_cells.csv
PFAD = "/Users/sven/Desktop/Udemy/progress/allen_cells.csv"
cells = pd.read_csv(PFAD)

# Table 1: one row per neuron (metadata)
zellen = cells[["specimen__id", "donor__id", "donor__species", "tag__dendrite_type",
                "structure__layer", "structure_parent__acronym"]].rename(columns={
    "specimen__id": "cell_id", "donor__id": "donor_id", "donor__species": "species",
    "tag__dendrite_type": "dendrite_type", "structure__layer": "layer",
    "structure_parent__acronym": "area"})
zellen["species"] = zellen["species"].replace({"Mus musculus": "mouse", "Homo Sapiens": "human"})

# Table 2: electrophysiology – only neurons with a measured firing rate
ephys = cells[cells["ef__avg_firing_rate"].notna()][["specimen__id", "ef__vrest", "ef__ri",
                                                      "ef__tau", "ef__avg_firing_rate"]].rename(columns={
    "specimen__id": "specimen", "ef__vrest": "vrest_mV", "ef__ri": "ri_MOhm",
    "ef__tau": "tau_ms", "ef__avg_firing_rate": "firing_rate_Hz"})

# Table 3: human donors – one row per donor
spender = cells[cells["donor__species"] == "Homo Sapiens"][["donor__id", "donor__age",
                "donor__sex", "donor__disease_state"]].drop_duplicates().rename(columns={
    "donor__id": "id", "donor__age": "age", "donor__sex": "sex", "donor__disease_state": "disease"})

# Table 4: a small lookup table of brain areas (written by hand, not complete!)
areale = pd.DataFrame({
    "area": ["VISp", "VISl", "VISpm", "VISal", "VISam", "MTG", "ITG", "TemL",
             "MFG", "SFG", "IFG", "FroL", "MOp"],
    "area_name": ["Primary visual area", "Lateral visual area", "Posteromedial visual area",
                  "Anterolateral visual area", "Anteromedial visual area", "Middle temporal gyrus",
                  "Inferior temporal gyrus", "Temporal lobe", "Middle frontal gyrus",
                  "Superior frontal gyrus", "Inferior frontal gyrus", "Frontal lobe", "Primary motor area"],
    "system": ["visual", "visual", "visual", "visual", "visual", "temporal", "temporal", "temporal",
               "frontal", "frontal", "frontal", "frontal", "motor"],
})

zellen.head()

,cell_id,donor_id,species,dendrite_type,layer,area
0,525011903,524848408,human,spiny,3,FroL
1,565871768,563340157,mouse,aspiny,5,VISp
2,469801138,319194354,mouse,aspiny,4,VISp
3,528642047,528574320,human,aspiny,5,MTG
4,605889373,605622695,mouse,aspiny,2/3,VISp


---
## Tasks

**1 ⭐ Get to know the tables:** Print the `shape` of all four tables. 🧠 (a) Why does `ephys` have fewer rows than `zellen`? (b) Why does `spender` have only 42 rows?

In [2]:
zellen.info()
print('-_-' *40)
ephys.info()
print('-_-' *40)
spender.info()
print('-_-' *40)
areale.info()

# ephys has fewer rows than zellen, 157 neurons have no firing rate value, so they were filtered out
# spender only has 42 rows, probably because each spender donated more than one intact cell

<class 'pandas.DataFrame'>
RangeIndex: 2333 entries, 0 to 2332
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   cell_id        2333 non-null   int64
 1   donor_id       2333 non-null   int64
 2   species        2333 non-null   str  
 3   dendrite_type  2333 non-null   str  
 4   layer          2333 non-null   str  
 5   area           2333 non-null   str  
dtypes: int64(2), str(4)
memory usage: 109.5 KB
-_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_--_-
<class 'pandas.DataFrame'>
Index: 2176 entries, 0 to 2332
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   specimen        2176 non-null   int64  
 1   vrest_mV        2176 non-null   float64
 2   ri_MOhm         2176 non-null   float64
 3   tau_ms          2176 non-null   float64
 4   firing_rate_Hz  2176 non-null   flo

**2 ⭐ `concat`:** Split `zellen` into `maus` (mouse) and `mensch` (human). Then stack them again with `pd.concat` and `keys=["mouse", "human"]`. Check with `len()` that no row got lost. 🧠 What kind of index does the result have?

In [3]:
human = zellen[zellen['species'] == 'human']
mouse = zellen[zellen['species'] == 'mouse']
fusion = pd.concat([human, mouse], keys = ['human', 'mouse'])
print(len(fusion))
fusion


2333


cell_id   donor_id species dendrite_type layer  area
human 0     525011903  524848408   human         spiny     3  FroL
      3     528642047  528574320   human        aspiny     5   MTG
      5     537256313  536912860   human         spiny     4   MTG
      13    519832676  518641172   human         spiny     3   MTG
      24    596020931  595954915   human        aspiny     4   MTG
...               ...        ...     ...           ...   ...   ...
mouse 2327  512322162  511856495   mouse         spiny    6b  VISp
      2329  488473535  487607572   mouse        aspiny   2/3  VISp
      2330  467003163  322489498   mouse         spiny     4  VISp
      2331  572609946  570930952   mouse        aspiny     4  VISp
      2332  464212183  312883165   mouse        aspiny     5  VISp

[2333 rows x 6 columns]

**3 ⭐⭐ Left join with the area table:** Add `area_name` and `system` to every neuron with a **left join** on `area`. Store it as `mit_areal`. (a) Does it still have 2,333 rows? (b) For how many neurons is `area_name` missing? (c) Which areas are missing most often in the lookup table?

In [4]:
mit_areal = zellen.merge(areale, how = 'left', on ='area')
print(len(mit_areal)) # yes mit_areal still has 2333 rows
print(mit_areal['area_name'].isna().sum()) # for 153 neurons area_name is missing

fehlt = mit_areal[mit_areal['area_name'].isna()]
fehlt['area'].value_counts()



2333
153


area
VISpor     25
VISpl      21
VISli      19
RSPagl     19
RSPd       19
AnG        16
PLP         6
SSp-bfd     5
VISa        5
SSp-tr      4
VISrl       3
TEa         2
AUDpo       2
AUDp        2
RSPv        2
SSp-n       1
SSp         1
SSp-un      1
Name: count, dtype: int64

**4 ⭐⭐ Inner join:** Do the same merge with `how="inner"`. How many rows are left? Explain the difference to task 3 in one sentence.

In [5]:
zellen_inner = zellen.merge(areale, how = 'inner', on = 'area')

fehlt_neu = zellen_inner[zellen_inner['area_name'].isna()]
fehlt = mit_areal[mit_areal['area_name'].isna()]
print(len(fehlt))
print(len(fehlt_neu))
# in Task 3, all rows are kept, also those, which do not share an 'area-value'. In Task 4, only the rows are kept, that share a common 'area-value'


153
0


**5 ⭐⭐ Different key names:** Add the electrophysiology to `zellen`. The key is called `cell_id` in `zellen` but `specimen` in `ephys` → use `left_on` and `right_on` with `how="left"`. Store it as `voll`. (a) How many rows? (b) How many neurons have no `firing_rate_Hz`? (c) Remove the now duplicate column `specimen`.

In [6]:
voll = zellen.merge(ephys, left_on = 'cell_id', right_on = 'specimen', how = 'left')
print(len(voll)) # 'voll' has 2333 rows
fehlt_ephys=voll[voll['firing_rate_Hz'].isna()]
print(len(fehlt_ephys)) # 157 cells do not have a 'firing_rate_Hz' - value
voll = voll.drop('specimen', axis='columns')
voll

2333
157


,cell_id,donor_id,species,dendrite_type,layer,area,vrest_mV,ri_MOhm,tau_ms,firing_rate_Hz
0,525011903,524848408,human,spiny,3,FroL,-70.561035,159.531131,21.181026,13.572511
1,565871768,563340157,mouse,aspiny,5,VISp,NaN,NaN,NaN,NaN
2,469801138,319194354,mouse,aspiny,4,VISp,-72.042976,65.624997,4.121689,79.952029
3,528642047,528574320,human,aspiny,5,MTG,-69.209610,201.562688,12.614641,77.445126
4,605889373,605622695,mouse,aspiny,2/3,VISp,-72.712036,102.968708,5.166574,12.666099
...,...,...,...,...,...,...,...,...,...,...
2328,601958555,601901227,human,spiny,4,MTG,NaN,NaN,NaN,NaN
2329,488473535,487607572,mouse,aspiny,2/3,VISp,-78.499512,63.124999,5.068471,47.867248
2330,467003163,322489498,mouse,spiny,4,VISp,-71.000069,130.312577,11.620790,16.498150
2331,572609946,570930952,mouse,aspiny,4,VISp,-67.397964,126.919538,7.600384,81.532817


**6 🔍 Where did the neurons go?** Someone wanted all neurons with their electrophysiology and wrote:

```python
zellen.merge(ephys, left_on="cell_id", right_on="specimen")
```

The result has only 2,176 rows. No error message. What happened? Fix it so all 2,333 neurons stay.

In [7]:
zellen.merge(ephys, how = 'left',  left_on="cell_id", right_on="specimen")

,cell_id,donor_id,species,dendrite_type,layer,area,specimen,vrest_mV,ri_MOhm,tau_ms,firing_rate_Hz
0,525011903,524848408,human,spiny,3,FroL,525011903.0,-70.561035,159.531131,21.181026,13.572511
1,565871768,563340157,mouse,aspiny,5,VISp,NaN,NaN,NaN,NaN,NaN
2,469801138,319194354,mouse,aspiny,4,VISp,469801138.0,-72.042976,65.624997,4.121689,79.952029
3,528642047,528574320,human,aspiny,5,MTG,528642047.0,-69.209610,201.562688,12.614641,77.445126
4,605889373,605622695,mouse,aspiny,2/3,VISp,605889373.0,-72.712036,102.968708,5.166574,12.666099
...,...,...,...,...,...,...,...,...,...,...,...
2328,601958555,601901227,human,spiny,4,MTG,NaN,NaN,NaN,NaN,NaN
2329,488473535,487607572,mouse,aspiny,2/3,VISp,488473535.0,-78.499512,63.124999,5.068471,47.867248
2330,467003163,322489498,mouse,spiny,4,VISp,467003163.0,-71.000069,130.312577,11.620790,16.498150
2331,572609946,570930952,mouse,aspiny,4,VISp,572609946.0,-67.397964,126.919538,7.600384,81.532817


**7 ⭐⭐ Outer join with `indicator`:** Merge `zellen` and `areale` with `how="outer"` and `indicator=True`. The new column `_merge` tells you where each row comes from. Count its values. 🧠 What does `right_only` mean here – and which area is it?

In [8]:
zellen_outer = zellen.merge(areale, how = 'outer', on = 'area', indicator=True)
print(zellen_outer['_merge'].value_counts())
zellen_outer[zellen_outer['_merge'] == 'right_only'] # Primary motor area

_merge
both          2180
left_only      153
right_only       1
Name: count, dtype: int64


,cell_id,donor_id,species,dendrite_type,layer,area,area_name,system,_merge
88,NaN,NaN,NaN,NaN,NaN,MOp,Primary motor area,motor,right_only


**8 ⭐⭐ Human donors:** Add the donor information to `mensch` (from task 2). Key: `donor_id` in `mensch`, `id` in `spender`. Store it as `m`. (a) How many neurons per `disease`? (b) How many **donors** per `sex`? 🧠 For (b), which table do you need – `m` or `spender` – and why?

In [9]:
mensch = zellen[zellen['species'] == 'human']
m = mensch.merge(spender, left_on = 'donor_id', right_on = 'id', how = 'left')
disease = m.groupby('disease').size()
print(m['disease'].value_counts())

spender['sex'].value_counts()
# spender

disease
epilepsy    350
tumor        63
Name: count, dtype: int64


sex
Male      23
Female    19
Name: count, dtype: int64

**9 ⭐⭐ `join` with the index:** Set `cell_id` as index of `zellen` and `specimen` as index of `ephys`, then combine them with `.join()`. How many rows? 🧠 Which kind of join does `.join()` do by default?

In [10]:
zi = zellen.set_index('cell_id')
eph = ephys.set_index('specimen')
combined = zi.join(eph)
len(combined) # 2333 rows



2333

**10 ⭐⭐⭐ Everything together:** Build one table `alles` = `voll` (task 5) merged with `areale` (inner join on `area`). Then: (a) mean `firing_rate_Hz` per `system` and `dendrite_type`, as a table with `unstack`, rounded to 1 decimal. (b) A loop over `alles.groupby("system")` that prints the system, the number of neurons and the mean `tau_ms` (rounded to 1 decimal).

In [13]:
alles = zellen.merge(ephys, how = 'left', left_on = 'cell_id', right_on = 'specimen')
print(len(alles))
alles = alles.merge(areale, how = 'inner', on = 'area')
grouped = alles.groupby(['system', 'dendrite_type'])
table = grouped['firing_rate_Hz'].mean().round(1).unstack()
table

2333


dendrite_type,aspiny,sparsely spiny,spiny
system,,,
frontal,85.9,15.3,42.8
temporal,33.0,24.3,25.7
visual,47.3,24.3,13.6


2333


dendrite_type,aspiny,sparsely spiny,spiny
system,,,
frontal,85.9,15.3,42.8
temporal,33.0,24.3,25.7
visual,47.3,24.3,13.6


In [14]:
for name, gruppe in alles.groupby('system'):
    print(name, len(gruppe), round(gruppe['tau_ms'].mean(), 1))

frontal 56 19.3
temporal 335 26.9
visual 1789 18.0
frontal 56 19.3
temporal 335 26.9
visual 1789 18.0


---
✅ **Done?** Restart Kernel and Run All, then let me know! Next step: add it to your GitHub repository. 🚀